In [1]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


# Test with custom instructions, overriding the instruction in rag_helper

In [2]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [3]:
assistant.rag("How do I get a certificate?")
assistant.rag("Can I still join the course after it started?")

'Yes — you can still join the course after it has started. You can also start whenever you want. If you want a certificate, though, you need to finish with a live cohort and submit your project while submissions are still being accepted.'

# RAG on typo

In [5]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your operating system:\n   - macOS: download the `.pkg`\n   - Windows: download the `.msi`\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and run:\n   ```bash\n   ollama run llama3\n   ```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\nTo test the local server, run:\n```bash\ncurl http://localhost:11434\n```\n\nYou should get a response like:\n```json\n{"models": [...]}\n```'

In [6]:
assistant.rag("How do I run Olama locally?")

'I don’t see any context about **Olama** specifically.\n\nIf you meant **running the course locally**, the FAQ says you can do that if you’re comfortable setting up **Python, `uv`, Jupyter, Docker, and any other tools needed for the module**. It also says to **document your setup and keep it reproducible**.\n\nIf you meant something else by “Olama,” please clarify the name and I’ll try to answer from the FAQ context.'

In [7]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Probably, yes — but it depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, I can help you figure it out quickly. Just send me:\n- the course name\n- where it’s offered\n- any deadline or term dates you saw\n\nIf you’re asking someone directly, you could say:\n\n“Hi, I just discovered the course and I’m interested in joining. Is it still possible to enroll?”\n\nIf you want, I can also help you write a more formal or more casual message.'

# Agent approach

In [11]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join join course discovered"}', call_id='call_AtmMZHrzzsER4DI1KKJ0XI0r', name='search', type='function_call', id='fc_064ee26fa183194b006a399ee917dc81a2a80d1c39606dd383', namespace=None, status='completed')]

In [12]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [13]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [14]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

# Calculate costs

In [15]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


# Agent Loop

## A developer prompt

In [16]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

## Function-call helper

In [17]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

## The full agent loop

In [18]:
it = 1

while it < 10:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join.

If you want a certificate, make sure to submit your project while submissions are still open.


In [19]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1
    max_iterations = 10

    while it < max_iterations:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    # Return error if it reached max iterations without converging
    if it == max_iterations:
        return "Error: Unable to answer the question within the maximum number of iterations."

    return last_answer

In [20]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally install run local model Ollama"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - **macOS**: download the `.pkg` from https://ollama.com/download
   - **Windows**: download the `.msi` from https://ollama.com/download
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This will download the model and start a local chat interface.

3. **Check that the server is running**
   ```bash
   curl http://localhost:11434
   ```
   If it’s working, you should get a response showing available models/info.

4. **Use it from Python**
   Install the client:
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['message']['content'])
   ```

If y

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - **macOS**: download the `.pkg` from https://ollama.com/download\n   - **Windows**: download the `.msi` from https://ollama.com/download\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and start a local chat interface.\n\n3. **Check that the server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   If it’s working, you should get a response showing available models/info.\n\n4. **Use it from Python**\n   Install the client:\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection issue, restarting the server often helps:\n```bash\nnoh

In [21]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"Can I still join the course if I just discovered it? enrollment late join join course FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, the key thing is to submit your project while the course is still accepting submissions. Homework is not mandatory for the certificate, though it’s recommended.

If you’d like, I can also help with how to get started or explain the certificate requirements. Any other areas you want to explore?


'Yes — you can still join the course.\n\nIf you want a certificate, the key thing is to submit your project while the course is still accepting submissions. Homework is not mandatory for the certificate, though it’s recommended.\n\nIf you’d like, I can also help with how to get started or explain the certificate requirements. Any other areas you want to explore?'

# off-topic

In [22]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit what is it"}
function_call: search {"query":"queen's gambit course FAQ chess opening"}
function_call: search {"query":"gambit chess opening queen side definition"}
iteration #2...
ASSISTANT:
“Queen’s gambit” is a chess opening. It usually starts with:

1. d4 d5  
2. c4

White offers the c-pawn as a gambit to try to gain control of the center. If Black accepts, it’s called the **Queen’s Gambit Accepted**; if Black declines, it’s the **Queen’s Gambit Declined**.

If you want, I can also explain:
- why it’s called a gambit,
- the main ideas for White and Black,
- or the difference between Accepted and Declined.


'“Queen’s gambit” is a chess opening. It usually starts with:\n\n1. d4 d5  \n2. c4\n\nWhite offers the c-pawn as a gambit to try to gain control of the center. If Black accepts, it’s called the **Queen’s Gambit Accepted**; if Black declines, it’s the **Queen’s Gambit Declined**.\n\nIf you want, I can also explain:\n- why it’s called a gambit,\n- the main ideas for White and Black,\n- or the difference between Accepted and Declined.'

# Experiment on ToyAIKit

In [23]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [24]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

## Letting ToyAIKit generate the schema

In [25]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [26]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [27]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

## Chat interface and runner

In [28]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [29]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [30]:
result.cost

CostInfo(input_cost=Decimal('0.00179025'), output_cost=Decimal('0.0017235'), total_cost=Decimal('0.00351375'))

In [31]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Ollama run locally install local setup FAQ"}', call_id='call_EHWRgWxVKAIYmK3hnm5uKJ0D', name='search', type='function_call', id='fc_089005515f5a7637006a39a5a8d14c8191924732252f20dedf', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"Olama local run install FAQ"}', call_id='call_o4hncDNU

In [32]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


## Interactive chat

In [ ]:
runner.run()